# Clustering jerárquico — Ejemplos

**Módulo 3 — Modelado y evaluación de enfoques no supervisados · Curso Analítica de Datos**

Este notebook acompaña las diapositivas [`3.11_Clustering_Jerarquico.pdf`](3.11_Clustering_Jerarquico.pdf) y lleva a código lo que allí se explica.

## Contenido

1. **Particiones anidadas**: por qué una jerarquía no obliga a fijar K antes de construirla.
2. **El procedimiento aglomerativo, desde cero**: reproducimos el ejemplo de los cuatro animales con enlace simple.
3. **Criterios de enlace (simple, completo, promedio, Ward)**: comparamos los cuatro sobre el mismo par de grupos.
4. **Construcción de un dendrograma**: el ejemplo A, B, C, D de las diapositivas.
5. **Lectura y corte del dendrograma**: el ejemplo A...F con tres cortes distintos.
6. **Comparación con K-means**: aplicamos clustering jerárquico y K-means sobre el dataset Iris y comparamos qué tan bien cada uno recupera las especies reales.

> 💡 Este notebook **no requiere descargar nada ni tener cuenta de Kaggle** — los datos de ejemplo se escriben en el propio notebook e Iris viene incluido en `scikit-learn`.

In [ ]:
# Librerías que usaremos en todo el notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.datasets import load_iris
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score

pd.set_option('display.max_columns', None)
np.random.seed(42)

## 1. Particiones anidadas y número de grupos

A diferencia de K-means, el clustering jerárquico **no pide fijar K de antemano**. Construye una jerarquía completa de grupos dentro de grupos, y la partición final se obtiene después, eligiendo un **corte** (una altura) o un número de grupos deseado.

Con cuatro observaciones A, B, C, D, una posible secuencia de inclusiones es:

$$\{A\},\{B\},\{C\},\{D\} \;\to\; \{A,B\},\{C\},\{D\} \;\to\; \{A,B\},\{C,D\} \;\to\; \{A,B,C,D\}$$

Cada paso fusiona el par de grupos más cercano según el criterio de enlace elegido. Nunca se deshace una fusión: por eso se dice que las particiones están **anidadas**.

## 2. El procedimiento aglomerativo, desde cero

Reproducimos el ejemplo de las diapositivas: agrupar cuatro animales por una sola variable, su peso en kg.

| Animal | Peso (kg) |
|---|---|
| Conejo | 2 |
| Gato | 4 |
| Perro | 20 |
| Lobo | 24 |

Usamos **enlace simple**: la distancia entre dos grupos es la **menor** distancia entre cualquier par de sus miembros. El procedimiento aglomerativo:

1. Empieza con cada observación como su propio grupo.
2. Calcula las distancias entre todos los grupos.
3. Fusiona el par de grupos más cercano.
4. Actualiza las distancias y repite hasta que quede un solo grupo.

In [ ]:
animales = pd.DataFrame({'animal': ['Conejo', 'Gato', 'Perro', 'Lobo'], 'peso_kg': [2, 4, 20, 24]})
animales

In [ ]:
def enlace_simple_desde_cero(X, nombres):
    """Clustering jerárquico aglomerativo con enlace simple, implementado paso a paso."""
    n = len(X)
    clusters = {i: [i] for i in range(n)}
    activos = list(range(n))
    dist = squareform(pdist(X, metric='euclidean'))
    historial = []
    while len(activos) > 1:
        mejor, mejor_d = None, np.inf
        for i in range(len(activos)):
            for j in range(i + 1, len(activos)):
                ci, cj = activos[i], activos[j]
                # enlace simple: la distancia entre grupos es la menor entre sus miembros
                d = min(dist[a, b] for a in clusters[ci] for b in clusters[cj])
                if d < mejor_d:
                    mejor_d, mejor = d, (ci, cj)
        ci, cj = mejor
        nuevo_id = max(clusters.keys()) + 1
        clusters[nuevo_id] = clusters[ci] + clusters[cj]
        historial.append((
            {nombres[k] for k in clusters[ci]},
            {nombres[k] for k in clusters[cj]},
            mejor_d,
        ))
        activos.remove(ci); activos.remove(cj); activos.append(nuevo_id)
    return historial

X_animales = animales[['peso_kg']].values
historial = enlace_simple_desde_cero(X_animales, animales['animal'].values)

for grupo1, grupo2, d in historial:
    print(f'Se unen {grupo1} y {grupo2} a distancia {d}')

Las tres fusiones ocurren exactamente donde indican las diapositivas: **Conejo+Gato a distancia 2**, **Perro+Lobo a distancia 4**, y la unión final de los dos grupos a **distancia 16** — el mínimo entre las cuatro distancias cruzadas (18, 22, 16, 20).

Verifiquemos con la función `linkage` de `scipy`, que es la que usaremos en el resto del notebook en vez de reimplementar el algoritmo cada vez.

In [ ]:
Z_animales = linkage(X_animales, method='single')

# cada fila: [id_cluster_1, id_cluster_2, distancia_de_fusion, num_observaciones]
print(Z_animales)

fig, ax = plt.subplots(figsize=(5, 4))
dendrogram(Z_animales, labels=animales['animal'].values, ax=ax)
ax.set_ylabel('Distancia de enlace simple (kg)')
ax.set_title('Enlace simple — cuatro animales')
plt.tight_layout()
plt.show()

## 3. Criterios de enlace (simple, completo, promedio, Ward)

Ahora comparamos los **cuatro criterios de enlace** de las diapositivas sobre el mismo par de grupos:

- **Grupo A** = {Conejo (2 kg), Gato (4 kg)}
- **Grupo B** = {Perro (20 kg), Lobo (24 kg)}

Cada criterio responde de forma distinta a la pregunta "¿qué tan separados están estos dos grupos?".

In [ ]:
grupoA = np.array([2, 4])    # conejo, gato
grupoB = np.array([20, 24])  # perro, lobo

# 1. Simple: distancia entre los dos miembros más cercanos
simple = min(abs(a - b) for a in grupoA for b in grupoB)

# 2. Completo: distancia entre los dos miembros más lejanos
completo = max(abs(a - b) for a in grupoA for b in grupoB)

# 3. Promedio: promedia las cuatro distancias cruzadas
promedio = np.mean([abs(a - b) for a in grupoA for b in grupoB])

print(f'Simple   (gato-perro):   |4-20|  = {simple} kg')
print(f'Completo (conejo-lobo):  |2-24|  = {completo} kg')
print(f'Promedio (los 4 pares):            {promedio} kg')

In [ ]:
# 4. Ward: cuánto aumenta la dispersión (suma de cuadrados) al fusionar
centroA, centroB = grupoA.mean(), grupoB.mean()
ss_antes = ((grupoA - centroA) ** 2).sum() + ((grupoB - centroB) ** 2).sum()

combinado = np.concatenate([grupoA, grupoB])
centro_combinado = combinado.mean()
ss_despues = ((combinado - centro_combinado) ** 2).sum()

incremento = ss_despues - ss_antes
altura_ward = np.sqrt(2 * incremento)  # convención de scipy para la altura del dendrograma

print(f'Centros antes de fusionar: {centroA} kg y {centroB} kg')
print(f'Suma de cuadrados antes:   {ss_antes} kg^2')
print(f'Centro combinado:          {centro_combinado} kg')
print(f'Suma de cuadrados después: {ss_despues} kg^2')
print(f'Incremento de dispersión:  {incremento} kg^2')
print(f'Altura de Ward (sqrt(2*incremento)): {altura_ward:.2f}')

Los cuatro valores coinciden con las diapositivas: **simple = 16**, **completo = 22**, **promedio = 19**, y **Ward**: dispersión antes = 10 kg², después = 371 kg², incremento = 361 kg², altura ≈ **26.87**.

Nótese que Ward no se mide en las mismas unidades que los otros tres — por eso su "altura" no es directamente comparable con las distancias en kg de simple/completo/promedio.

Con los cuatro animales completos, `scipy` arma el dendrograma completo con cada criterio:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
nombres_metodo = {'single': 'Simple', 'complete': 'Completo', 'average': 'Promedio', 'ward': 'Ward'}

for ax, metodo in zip(axes.ravel(), ['single', 'complete', 'average', 'ward']):
    Zm = linkage(X_animales, method=metodo)
    dendrogram(Zm, labels=animales['animal'].values, ax=ax)
    ax.set_title(f'{nombres_metodo[metodo]} (altura final: {Zm[-1, 2]:.2f})')

plt.tight_layout()
plt.show()

Las cuatro fusiones ocurren en el mismo orden (Conejo+Gato, Perro+Lobo, y luego los dos grupos entre sí) porque con solo cuatro puntos en una dimensión no hay ambigüedad sobre qué fusionar primero. Lo que cambia entre criterios es la **altura** a la que ocurre la última unión — eso es justamente lo que ilustra la pregunta de la diapositiva: "mismos datos, mismo orden de fusión, distintas alturas".

## 4. Construcción de un dendrograma

Reproducimos el segundo ejemplo de las diapositivas: cuatro puntos en una sola variable, A=1, B=2, C=8, D=10, con enlace simple y distancia euclídea $d(x,y) = |x-y|$.

In [ ]:
puntos = pd.DataFrame({'punto': ['A', 'B', 'C', 'D'], 'x': [1, 2, 8, 10]})
Z_abcd = linkage(puntos[['x']].values, method='single')

print(Z_abcd)
print('\nAlturas de fusión:', sorted(Z_abcd[:, 2]))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
dendrogram(Z_abcd, labels=puntos['punto'].values, ax=ax)
ax.set_ylabel('Distancia de enlace simple')
ax.set_title('A, B, C, D — enlace simple')
plt.tight_layout()
plt.show()

Tres fusiones, igual que en las diapositivas:

1. **A y B** se unen primero, a distancia $|1-2| = 1$.
2. **C y D** se unen, a distancia $|8-10| = 2$.
3. Los grupos **{A,B}** y **{C,D}** se unen al final. Con enlace simple, la distancia entre grupos es la menor de las distancias cruzadas: $\min(|1-8|, |1-10|, |2-8|, |2-10|) = \min(7, 9, 6, 8) = 6$ — el par más cercano entre los dos grupos es B-C.

Las alturas de fusión que calculamos son exactamente **1, 2 y 6**, igual que en la diapositiva.

## 5. Lectura y corte del dendrograma

Un mismo dendrograma admite varias particiones, según dónde se corte. Extendemos el ejemplo anterior con dos puntos más: E=20, F=23.

In [ ]:
puntos2 = pd.DataFrame({'punto': ['A', 'B', 'C', 'D', 'E', 'F'], 'x': [1, 2, 8, 10, 20, 23]})
Z_abcdef = linkage(puntos2[['x']].values, method='single')

print(Z_abcdef)
print('\nAlturas de fusión:', sorted(Z_abcdef[:, 2]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, h, titulo in zip(axes, [4, 8, 12], ['3 grupos (corte h=4)', '2 grupos (corte h=8)', '1 grupo (corte h=12)']):
    dendrogram(Z_abcdef, labels=puntos2['punto'].values, ax=ax, color_threshold=h)
    ax.axhline(h, color='purple', linestyle='--')
    ax.set_title(titulo)
    ax.set_ylabel('Altura de fusión')

plt.tight_layout()
plt.show()

In [ ]:
# fcluster(..., criterion='distance') asigna etiquetas de grupo cortando a una altura dada
for h in [4, 8, 12]:
    etiquetas = fcluster(Z_abcdef, t=h, criterion='distance')
    grupos = pd.DataFrame({'punto': puntos2['punto'], 'grupo': etiquetas})
    print(f'--- corte h={h}: {etiquetas.max()} grupo(s) ---')
    print(grupos.groupby('grupo')['punto'].apply(list).to_string())
    print()

Para leer un dendrograma: se traza un corte horizontal y se cuentan las ramas verticales que atraviesa.

- **h = 4** cruza 3 ramas → 3 grupos: {A,B}, {C,D}, {E,F}.
- **h = 8** cruza 2 ramas → 2 grupos: {A,B,C,D}, {E,F}. Entre h=6 y h=10 el dendrograma no cambia de forma, así que cualquier corte en ese rango da la misma partición de 2 grupos.
- **h = 12** cruza 1 rama → 1 solo grupo: todos juntos.

Al subir el corte, los grupos se van uniendo — nunca se separan. Por eso conviene elegir alturas **entre** fusiones y no justo sobre una unión, donde el resultado sería ambiguo.

## 6. Comparación con K-means

La tabla de las diapositivas resume las diferencias:

| Aspecto | Jerárquico | K-means |
|---|---|---|
| Salida | Estructura anidada (dendrograma) | Partición para K elegido (centroides) |
| Decisiones | Distancia + enlace + corte | K, inicialización, distancia/escala |
| Uso | Exploración multiescala | Particiones reproducibles para K fijo |

Para verlo con datos reales, aplicamos clustering jerárquico (con dos criterios de enlace distintos) y K-means sobre el dataset **Iris**, y comparamos qué tan bien cada partición recupera las tres especies reales — aunque ninguno de los dos algoritmos usa esa etiqueta para agrupar, solo la usamos después para evaluar.

In [ ]:
iris = load_iris()
X_iris = iris.data
y_especie = iris.target

agg_ward = AgglomerativeClustering(n_clusters=3, linkage='ward').fit(X_iris)
agg_simple = AgglomerativeClustering(n_clusters=3, linkage='single').fit(X_iris)
km_iris = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_iris)

# Adjusted Rand Index: mide qué tan parecida es una partición a la etiqueta real (1 = idéntica, 0 = azar)
ari_ward = adjusted_rand_score(y_especie, agg_ward.labels_)
ari_simple = adjusted_rand_score(y_especie, agg_simple.labels_)
ari_km = adjusted_rand_score(y_especie, km_iris.labels_)

print(f'ARI jerárquico (Ward):   {ari_ward:.3f}')
print(f'ARI jerárquico (simple): {ari_simple:.3f}')
print(f'ARI K-means:             {ari_km:.3f}')

print('\nTamaños de grupo — Ward:  ', np.bincount(agg_ward.labels_))
print('Tamaños de grupo — simple:', np.bincount(agg_simple.labels_))
print('Tamaños de grupo — K-means:', np.bincount(km_iris.labels_))

Dos observaciones:

1. **Ward y K-means obtienen un desempeño casi idéntico** (ARI ≈ 0.73 ambos) — no es casualidad: Ward minimiza el mismo tipo de dispersión interna (suma de cuadrados) que K-means, solo que de forma jerárquica en vez de iterativa.
2. **Enlace simple se comporta mucho peor** (ARI ≈ 0.56) y produce un grupo de tamaño 2: esto es el fenómeno de **encadenamiento** (*chaining*) — enlace simple fusiona en cuanto encuentra un par cercano, sin importar la forma global del grupo, así que una cadena de puntos intermedios puede terminar uniendo observaciones que en realidad pertenecen a especies distintas, dejando un grupo casi vacío.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
Z_iris = linkage(X_iris, method='ward')

# color_threshold justo debajo de la penúltima fusión resalta los 3 grupos finales
dendrogram(Z_iris, no_labels=True, color_threshold=Z_iris[-3, 2] + 0.01, ax=ax)
ax.set_ylabel('Altura de fusión (Ward)')
ax.set_title('Dendrograma de Iris (enlace Ward, 150 observaciones)')
plt.tight_layout()
plt.show()

A diferencia de K-means, aquí no tuvimos que decidir K de antemano para construir el dendrograma — la estructura completa ya estaba disponible, y "cortar en 3 grupos" fue una decisión que tomamos **después**, mirando la jerarquía. Esa es la principal ventaja exploratoria del clustering jerárquico: permite comparar varias particiones (2 grupos, 3 grupos, 4 grupos...) sin rehacer el cálculo, algo que con K-means exigiría volver a correr el algoritmo completo para cada valor de K.

---

## Actividad abierta para discutir en clase

1. En el ejemplo de los animales, ¿por qué el criterio de enlace **completo** da una distancia mayor que **promedio**, y este a su vez mayor que **simple**? ¿Es esto una coincidencia de estos datos o siempre se cumple ese orden?
2. En el dendrograma de Iris, ¿qué pasaría con el ARI de enlace simple si en vez de 3 grupos pidiéramos 2? ¿Y con 10 grupos? Pruébenlo cambiando `n_clusters` en el código.
3. La tabla de la diapositiva dice que K-means ofrece "particiones reproducibles para K fijo". ¿En qué sentido el clustering jerárquico con un criterio de enlace fijo (por ejemplo, Ward) también es reproducible? ¿Qué sí cambia de una corrida a otra en K-means que no cambia en jerárquico?
4. ¿En qué situación preferirían explorar con un dendrograma antes de comprometerse con un K fijo para K-means?